## 逐个输入多组学信息

| 模态  | 特征           | 处理方式      |
| --- | ------------ | --------- |
| 模态1 | DTC_min + 大小 + 钙化类型 | 数值 + 类别编码 |
| 模态2 | TPO          | 数值（缺失填充）  |
| 模态3 | BRAF         | 类别（突变/野生） |


① 纯影像（baseline）

② 影像 + 单个组学（3组）

③ 影像 + 全部组学（1组）

In [ ]:
# region 数据预处理
import os
import pandas as pd
import numpy as np
import cv2
import warnings

from skimage.feature import local_binary_pattern, hog, graycomatrix, graycoprops
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# 忽略警告
warnings.filterwarnings('ignore')

# ==========================================
# 路径配置
# ==========================================
PATH_NO = r'F:\数据集\淋巴结转移\jpg\ROI无转移'
PATH_YES = r'F:\数据集\淋巴结转移\jpg\ROI有转移'
LABEL_FILE = r'F:\数据集\淋巴结转移\complete_information_last.xlsx'

# 组学特征增强权重
ALPHA = 5

# ==========================================
# 1. 图像特征提取 (Baseline)
# ==========================================
def extract_radiomics_features(img_path):
    # 读取中文路径图片
    img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    img = cv2.resize(img, (128, 128))

    # 增强对比度
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)

    # 统计特征
    flat = img.flatten()
    stats = [np.mean(flat), np.std(flat),
             pd.Series(flat).skew(), pd.Series(flat).kurt()]

    # 纹理特征 (GLCM)
    distances = [1, 3]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(img, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)
    
    glcm_feats = []
    for prop in ['contrast','dissimilarity','homogeneity','energy','correlation','ASM']:
        glcm_feats.extend(graycoprops(glcm, prop).flatten())

    # LBP
    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0,11),
                              range=(0,10), density=True)

    # HOG
    hog_feat = hog(img, orientations=8,
                   pixels_per_cell=(32,32),
                   cells_per_block=(1,1))

    return np.concatenate([stats, glcm_feats, hist_lbp, hog_feat])


# ==========================================
# 2. 临床/组学特征提取 (修改后：3个模态)
# ==========================================
def extract_clinical_features(row):
    # --- 模态1: 距离 + 大小 + 回声 ---
    distance = row['距离/mm']
    size = row['大小']
    echo_map = {'极低回声': 0, '低回声': 1, '混合回声': 2}
    echo = echo_map.get(row['结节回声'], -1) # 默认-1
    m1 = [distance, size, echo]

    # --- 模态2: TPO (数值) ---
    m2 = [row['TPO']]

    # --- 模态3: BRAF (类别) ---
    braf = row['BRAF V600']
    if pd.isna(braf):
        braf_val = -1
    else:
        braf_str = str(braf)
        if '突变' in braf_str:
            braf_val = 1
        elif '野生' in braf_str:
            braf_val = 0
        else:
            braf_val = -1
    m3 = [braf_val]

    return m1, m2, m3


# ==========================================
# 3. 数据读取与预处理
# ==========================================
df = pd.read_excel(LABEL_FILE)

X_img, X_m1, X_m2, X_m3, y = [], [], [], [], []

print("正在提取特征，请稍候...")

for _, row in df.iterrows():
    img_name = str(row['image_name'])
    label = row['label']
    
    # 确定路径
    img_path = os.path.join(PATH_NO if label == 0 else PATH_YES, img_name)

    if not os.path.exists(img_path):
        continue

    # 提取影像特征
    img_feat = extract_radiomics_features(img_path)
    if img_feat is None:
        continue

    # 提取组学特征 (3组)
    m1, m2, m3 = extract_clinical_features(row)

    X_img.append(img_feat)
    X_m1.append(m1)
    X_m2.append(m2)
    X_m3.append(m3)
    y.append(label)

# 转换为Numpy数组
X_img = np.array(X_img)
X_m1 = np.array(X_m1, dtype=float)
X_m2 = np.array(X_m2, dtype=float)
X_m3 = np.array(X_m3, dtype=float)
y = np.array(y)

# 缺失值填充函数 (中值填充)
def fill_nan(X):
    X_copy = X.copy()
    for i in range(X_copy.shape[1]):
        col = X_copy[:, i]
        if np.isnan(col).all():
            X_copy[:, i] = 0
        else:
            median = np.nanmedian(col)
            X_copy[np.isnan(col), i] = median
    return X_copy

X_m1 = fill_nan(X_m1)
X_m2 = fill_nan(X_m2)
X_m3 = fill_nan(X_m3)


# ==========================================
# 4. 实验设置
# ==========================================
# 重新定义的实验组合
experiments = {
    "Image (Baseline)": X_img,
    "Image + M1 (Distance/Size/Echo)": np.concatenate([X_img, X_m1], axis=1),
    "Image + M2 (TPO)": np.concatenate([X_img, X_m2], axis=1),
    "Image + M3 (BRAF)": np.concatenate([X_img, X_m3], axis=1),
    "Image + All Omics": np.concatenate([X_img, X_m1, X_m2, X_m3], axis=1)
}

models = {
    "RF": RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42),
    "SVM": SVC(probability=True, class_weight='balanced', kernel='rbf'),
    "XGB": XGBClassifier(n_estimators=100, max_depth=4, eval_metric='logloss', random_state=42),
    "LR": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "KNN": KNeighborsClassifier(5)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ==========================================
# 5. 训练与评估
# ==========================================
for exp_name, X_full in experiments.items():

    print(f"\n" + "="*50)
    print(f"正在进行实验: {exp_name}")
    print("="*50)

    for model_name, model in models.items():

        aucs, f1s, sens, spes = [], [], [], []

        for tr_idx, te_idx in skf.split(X_full, y):
            X_tr_all, X_te_all = X_full[tr_idx], X_full[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            # --- 内部特征处理流程 ---
            # 1. 拆分影像与组学（用于分别标准化和特征选择）
            img_dim = X_img.shape[1]
            X_tr_img = X_tr_all[:, :img_dim]
            X_te_img = X_te_all[:, :img_dim]
            
            X_tr_clin = X_tr_all[:, img_dim:]
            X_te_clin = X_te_all[:, img_dim:]

            # 2. 影像特征预处理 (标准化 + 特征降维)
            sc1 = StandardScaler()
            X_tr_img = sc1.fit_transform(X_tr_img)
            X_te_img = sc1.transform(X_te_img)

            sel = SelectKBest(f_classif, k=min(30, X_tr_img.shape[1]))
            X_tr_img = sel.fit_transform(X_tr_img, y_tr)
            X_te_img = sel.transform(X_te_img)

            # 3. 组学特征预处理 (标准化 + 权重增强)
            if X_tr_clin.shape[1] > 0:
                sc2 = StandardScaler()
                X_tr_clin = sc2.fit_transform(X_tr_clin)
                X_te_clin = sc2.transform(X_te_clin)

                # 应用权重 ALPHA
                X_tr_clin *= ALPHA
                X_te_clin *= ALPHA
                
                # 重新拼接
                X_tr_final = np.concatenate([X_tr_img, X_tr_clin], axis=1)
                X_te_final = np.concatenate([X_te_img, X_te_clin], axis=1)
            else:
                X_tr_final = X_tr_img
                X_te_final = X_te_img

            # 4. 训练与预测
            model.fit(X_tr_final, y_tr)
            y_pred = model.predict(X_te_final)
            y_prob = model.predict_proba(X_te_final)[:, 1]

            # 5. 指标计算
            tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
            
            aucs.append(roc_auc_score(y_te, y_prob))
            f1s.append(f1_score(y_te, y_pred))
            sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            spes.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

        # 输出平均结果
        print(f"{model_name:<5} | "
              f"AUC {np.mean(aucs):.3f}±{np.std(aucs):.3f} | "
              f"F1 {np.mean(f1s):.3f}±{np.std(f1s):.3f} | "
              f"Sen {np.mean(sens):.3f} | "
              f"Spe {np.mean(spes):.3f}")

print("\n所有实验已完成。")